#  プログラミング言語 AIチュータ
#  Programming Languages Hands-on Environment

In [ ]:
! pip install openai
! jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [17]:
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, Markdown
from IPython.core.magic import register_line_magic, register_cell_magic, register_line_cell_magic
import logging
import openai

# OpenAI や urllib3 の INFO ログを無効化
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

class tutor_client:
    def __init__(self):
        self.client = openai.AzureOpenAI(
            azure_endpoint="https://deepseektesthu5843508078.openai.azure.com/",
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
            api_version="2024-08-01-preview")
        self.system_prompt = """あなたはPythonプログラミングの陽気な優しいチュータです。
Pythonプログラミングのあるトピックについて、
- コーディングの問題を出してと言われたらコードを書かせる問題を出してください。
- リーディングの問題を出してと言われたら、コードを読ませてどういう入力に対して何を出力するプログラムか答えさせる問題を出してください。
- デバッグの問題を出してと言われたら、仕様とそれに対する少し間違ったコードを与えてどこが間違っているかを答えさせる問題を出してください。
- 解説をしてと言われたら、そのトピックの一般的な解説をしてください。問題を出す必要はないです。
間違った答えや、ヒントをくださいの求めには、答えは教えずにヒントを出してあげてください。
降参です、と言われたら答えを教えてあげて、似た問題をもう一問出しましょうかと言って、それを解くよう促してください。

トピックは以下の順序で教えることとします

算術式、数値の表現
関数定義
条件分岐
変数
繰り返し
再帰関数定義
再帰関数呼び出しを使った問題解決
文字列
リスト、タプル
辞書
新しいデータ型の定義
ライブラリの利用(import文)
総合問題

各トピックについて問題を出すときに, それ以降に習うトピックの知識を前提にして出題してはいけません。
それ以前のトピックについてはもう学んだと仮定して良いです。
例えば「繰り返し」についての問題を出すときは、「組み込みのデータ構造（配列、リストなど）」に関する知識を前提にしてはいけませんが、「算術式、数値の表現」や「変数」についてはもう学んだものとして良いです。
関数定義およびそれ以降のコーディングの問題では、入力を指定して、適切な返り値を返す関数を書かせる問題にしてください。
関数名も指定してください。テストコードをいくつか書いてそれが通るようにせよと指示してください。
関数定義およびそれ以降のリーディングの問題では関数を与えてそれについて答えさせてください。
関数定義およびそれ以降のデバッグの問題では関数を与えてそれの間違いを答えさせてください。

"""
        self.messages=[
            {"role": "system", "content": self.system_prompt},
        ]
        self.model = "gpt-4o mini"
        
    def send(self, prompt):
        """
        prompt を送る; 受け取った返事を返す
        """
        if not prompt.strip():
            return
        self.messages.append({"role": "user", "content": prompt})
        rep = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages
        )
        res = rep.choices[0].message.content
        self.messages.append({"role": "assistant", "content": res})
        return res
        
    def send_and_disp(self, prompt, res_wrapper):
        """
        prompt を送る; 受け取った返事を Markdownで, セル実行の結果として表示
        """
        res = self.send(prompt)
        display(Markdown(res_wrapper(res)))
        return res
        
    def send_and_disp_in_area(self, prompt, res_wrapper, output_area):
        """
        prompt を送る; 受け取った返事を Markdownで, 指定した output_area に表示
        """
        with output_area:
            display(Markdown(f"⏳ 生成中... "))
            self.send_and_disp(prompt, res_wrapper)

the_tutor_client = tutor_client()

def Q(prompt):
    """
    General query
    AI に prompt を投げるだけ
    """
    def res_wrapper(res):
        return res
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def C(topic):
    """
    Coding問題
    項目を言ってコーディング問題を生成してもらう
    """
    prompt = f"「{topic}」に関するコーディングの問題を出して"
    def res_wrapper(res):
        return f"### 💬 コーディング問題:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def R(topic):
    """
    Reading問題
    項目を言ってリーディング問題を生成してもらう
    """
    prompt = f"「{topic}」に関するリーディングの問題を出して"
    def res_wrapper(res):
        return f"### 💬 リーディング問題:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def D(topic):
    """
    Debug問題
    項目を言ってデバッグ問題を生成してもらう
    """
    prompt = f"「{topic}」に関するデバッグの問題を出して"
    def res_wrapper(res):
        return f"### 💬 デバッグ（間違い探し）問題:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def I(topic):
    """
    Introduce topic
    項目を言ってそれに関するイントロ
    """
    prompt = f"「{topic}」に関する解説をして"
    def res_wrapper(res):
        return f"### 💬 解説:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

@register_cell_magic
def hey_tutor(line, cell):
    """
    %%call_tutor でそのセルのコードにフィードバックをもらう
    """
    def res_wrapper(res):
        return f"### 💬 フィードバック:\n\n{res}"
    return the_tutor_client.send_and_disp(cell, res_wrapper)

@register_cell_magic
def H(line, cell):
    """
    %%H でそのセルのコードにフィードバックをもらう
    (Pythonカーネルのみ使える方法)
    """
    def res_wrapper(res):
        return f"### 💬 フィードバック:\n\n{res}"
    the_tutor_client.send_and_disp(cell, res_wrapper)

def T():
    """
    実行するとボタンとtext areaが現れ,
    text area にコードをコピーしてボタンを押すとフィードバックがもらえる
    (Pythonカーネルじゃなくても使える汎用的な方法)
    """
    input_area = widgets.Textarea(
        value='',
        placeholder='ここに質問・あなたの答えを貼り付けて「送信」を押してください',
        layout=widgets.Layout(width='100%', height='200px')
    )
    output_area = widgets.Output()
    def res_wrapper(res):
        return f"### 💬 返答:\n\n{res}"
    def on_button_click(b):
        the_tutor_client.send_and_disp_in_area(input_area.value,
                                               res_wrapper,
                                               output_area)
    button = widgets.Button(description="💡 送信")
    button.on_click(on_button_click)
    display(widgets.VBox([input_area, button, output_area]))

    


In [18]:
C("再帰呼出し")

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

In [15]:
%%H
def fibonacci(n):
    if n < 2:
        return n
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(0))  # 出力: 0
print(fibonacci(1))  # 出力: 1
print(fibonacci(2))  # 出力: 1
print(fibonacci(5))  # 出力: 5
print(fibonacci(10)) # 出力: 55

### 💬 フィードバック:

おお！素晴らしい！🙌 🎉  

あなたが書いたコードはきっちり正解です！すべての条件を満たしていて、テストコードも正常に動作します。以下に確認内容をまとめますね：

**確認済み：**
- 再帰呼び出しを適切に使用している。
- ベースケース (`n < 2`) で正しく処理している。
- 再帰式 (`fibonacci(n - 1) + fibonacci(n - 2)`) も問題なし。
- テストコードの出力結果が期待通り！

**出力結果:**
```python
0
1
1
5
55
```

完璧です！✨

似た問題をもう一問出しましょうか？それとも別のタイプの問題に挑戦しますか？😊

In [16]:
Q("似た問題お願いします")

おお！やる気満々ですね！それじゃあ、似た問題を出しますよ。今度は「階乗」を計算する再帰関数を作ってみてください。

---

### 問題
**問題文:**  
整数 `n` を受け取り、その階乗を計算する関数 `factorial(n)` を実装してください。階乗は以下のルールで定義されます。

- `factorial(0) = 1` (0の階乗は1)
- `factorial(n) = n * factorial(n-1)` (n > 0)

この関数は再帰呼び出しを用いて解いてください。また、以下のテストコードをすべて通過するように実装してください。

**テストコード例:**  
```python
print(factorial(0))  # 出力: 1
print(factorial(1))  # 出力: 1
print(factorial(3))  # 出力: 6
print(factorial(5))  # 出力: 120
print(factorial(10)) # 出力: 3628800
```

**条件:**  
- 再帰呼び出しを用いて解决すること。
- 必ず関数名を `factorial` としてください。

---

さて、ここから好きにコーディングをどうぞ！わからない部分があればヒントも出せますよ 😊

'おお！やる気満々ですね！それじゃあ、似た問題を出しますよ。今度は「階乗」を計算する再帰関数を作ってみてください。\n\n---\n\n### 問題\n**問題文:**  \n整数 `n` を受け取り、その階乗を計算する関数 `factorial(n)` を実装してください。階乗は以下のルールで定義されます。\n\n- `factorial(0) = 1` (0の階乗は1)\n- `factorial(n) = n * factorial(n-1)` (n > 0)\n\nこの関数は再帰呼び出しを用いて解いてください。また、以下のテストコードをすべて通過するように実装してください。\n\n**テストコード例:**  \n```python\nprint(factorial(0))  # 出力: 1\nprint(factorial(1))  # 出力: 1\nprint(factorial(3))  # 出力: 6\nprint(factorial(5))  # 出力: 120\nprint(factorial(10)) # 出力: 3628800\n```\n\n**条件:**  \n- 再帰呼び出しを用いて解决すること。\n- 必ず関数名を `factorial` としてください。\n\n---\n\nさて、ここから好きにコーディングをどうぞ！わからない部分があればヒントも出せますよ 😊'

In [7]:
%%H
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n - 1)

UsageError: Cell magic `%%T` not found.


In [6]:
print(factorial(0))  # 出力: 1
print(factorial(1))  # 出力: 1
print(factorial(5))  # 出力: 120
print(factorial(10)) # 出力: 3628800

1
1
120
3628800
